# Pertemuan 5 — Membersihkan Data, dengan Alasan
### Workbook belajar mandiri

**Exploratory Data Analysis (KMTI21133)**  
**Sub-CPMK5:** Mahasiswa mampu menerapkan teknik *data preparation* dan *data manipulation* dengan Python
disertai justifikasi setiap keputusan (C3).

---

> **Dosen masih berhalangan hadir.** Kelas didampingi asisten. Workbook ini dapat Anda selesaikan sendiri.

| | |
|---|---|
| **Perkiraan waktu** | 150 menit. Bagian 1–7 sekitar 60 menit, Bagian 8 sekitar 30 menit, Bagian 9–13 sekitar 35 menit, Bagian 14 sekitar 25 menit. |
| **Prasyarat** | Workbook pertemuan 4 sudah selesai, termasuk Bagian 10 (profil kualitas data tim). |
| **Umpan balik** | Sama seperti pekan lalu: sel `cek(...)` menjawab BENAR atau BELUM tanpa menampilkan kuncinya. |

### Yang Anda hasilkan di akhir sesi

**M2 — Data Quality Report (bobot 10%)**, terdiri atas tiga berkas:

1. `reports/data_quality_report.docx` — laporan temuan dan keputusan
2. `notebooks/02_cleaning.ipynb` — notebook pembersihan yang dapat dijalankan ulang
3. `cleaning_log.md` — catatan setiap keputusan beserta alasannya

Pakai `cek_m2.py` untuk memeriksa kelengkapan kiriman sebelum mengumpulkan.

---
## Bagian 0 — Persiapan

In [1]:
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 150)

FILE = 'transaksi_koperasi_2025.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

mentah = pd.read_csv(FILE)
print('Data mentah:', mentah.shape)

Saving transaksi_koperasi_2025.csv to transaksi_koperasi_2025.csv
Data mentah: (1543, 16)


In [2]:
_K = {
    'P5_baris_bersih': 'b3396bcb59',
    'P5_dibuang': '1a6562590e',
    'P5_kat': 'ef2d127de3',
    'P5_metode': '4b227777d4',
    'P5_anggota': '7c252ab334',
    'P5_total_num': 'b3396bcb59',
    'P5_curiga': '1a6562590e',
    'P5_rating_kosong': '25fc0e7096',
    'P5_bulan': '4b227777d4',
    'P5_langkah': 'e629fa6598'
}

_P = {
    'P5_baris_bersih': 'len() setelah seluruh langkah pembuangan baris dijalankan.',
    'P5_dibuang': 'Jumlah baris mentah dikurangi jumlah baris bersih.',
    'P5_kat': 'nunique() kolom kategori setelah diseragamkan dan sinonim disatukan.',
    'P5_metode': 'nunique() metode_bayar setelah placeholder menjadi NaN; pakai dropna().',
    'P5_anggota': 'nunique() id_anggota setelah strip.',
    'P5_total_num': 'Berapa baris yang total_bayar-nya sudah berupa angka (notna) setelah konversi.',
    'P5_curiga': 'Hitung baris yang selisih total_bayar dengan total_seharusnya lebih dari 1 rupiah.',
    'P5_rating_kosong': 'Setelah -999 diubah menjadi NaN, hitung isna() pada rating_layanan data bersih.',
    'P5_bulan': 'nunique() kolom bulan hasil dt.to_period(\'M\').',
    'P5_langkah': 'Berapa baris yang ada di cleaning log Anda.'
}

def _h(x):
    return hashlib.sha256(str(x).strip().lower().encode()).hexdigest()[:10]

def cek(tugas, jawaban):
    if tugas not in _K:
        print('Nama tugas tidak dikenal:', tugas); return
    if _h(jawaban) == _K[tugas]:
        print(f'BENAR   {tugas} = {jawaban}')
    else:
        print(f'BELUM   {tugas} = {jawaban}')
        print('        Petunjuk:', _P.get(tugas, '-'))

print('Alat pemeriksa siap.')

Alat pemeriksa siap.


---
## Bagian 1 — Pembersihan adalah keputusan analitis

Pekan lalu Anda menemukan belasan masalah. Hari ini setiap masalah itu menuntut sebuah **keputusan**,
dan hampir tidak ada yang punya jawaban tunggal.

Ambil satu contoh: 29 baris memiliki `jumlah` nol atau negatif. Ada empat pilihan, semuanya masuk akal:

| Pilihan | Masuk akal bila | Risikonya |
|---|---|---|
| Dibuang barisnya | Yakin itu kesalahan input | Kehilangan transaksi yang mungkin sah |
| Dijadikan kosong | Belum tahu penyebabnya | Analisis berkurang datanya |
| Diubah menjadi positif | Yakin itu retur yang salah tanda | Mengarang data bila dugaan salah |
| Dibiarkan, tetapi ditandai | Ingin menyerahkan keputusan ke tahap analisis | Harus diingat terus |

> **Yang dinilai bukan pilihan Anda, melainkan apakah Anda punya alasan dan menuliskannya.**
> Dua tim boleh mengambil keputusan berbeda pada data yang sama dan keduanya memperoleh nilai penuh,
> asalkan alasannya jelas. Tim yang membersihkan tanpa alasan tertulis tidak memperoleh nilai penuh
> meskipun kodenya rapi.

### Tiga aturan yang berlaku sepanjang pertemuan ini

1. **Jangan pernah menimpa data mentah.** `data/raw/` tidak pernah diubah. Hasil disimpan di `data/processed/`.
2. **Satu langkah, satu catatan.** Setiap perubahan masuk ke *cleaning log* saat itu juga, bukan di akhir.
3. **Menandai lebih aman daripada menebak.** Kalau Anda tidak tahu nilai yang benar, jangan mengarang.
   Jadikan kosong atau beri kolom penanda.

---
## Bagian 2 — Urutan pembersihan

Urutan bukan selera. Langkah yang dikerjakan terlalu awal akan membuat langkah berikutnya salah.

| Urutan | Langkah | Mengapa harus di sini |
|---|---|---|
| 1 | Perbaiki **tipe data** | Selama uang masih teks, tidak satu pun pemeriksaan angka bisa jalan |
| 2 | Buang **spasi** pada teks | Kalau tidak, penyeragaman kategori akan meleset |
| 3 | Ubah **placeholder** menjadi kosong | Kalau tidak, `-999` ikut terhitung sebagai nilai |
| 4 | **Seragamkan** kategori | Setelah spasi bersih, barulah sinonim terlihat |
| 5 | Tangani **duplikat** | Setelah teks seragam, duplikat nyaris baru bisa dikenali |
| 6 | Tangani **nilai mustahil** | Perlu tipe data yang benar untuk membandingkan |
| 7 | Tangani **nilai kosong** | Paling akhir, karena langkah 3 dan 6 menambah nilai kosong baru |
| 8 | Buat **kolom turunan** | Hanya bermakna di atas data yang sudah bersih |

> **Kesalahan urutan yang paling sering terjadi:** mengisi nilai kosong lebih dulu, lalu baru mengubah
> placeholder menjadi kosong. Hasilnya, `-999` sudah terlanjur ikut dirata-ratakan, dan nilai kosong yang
> baru muncul tidak pernah tertangani.

---
## Bagian 3 — Langkah 1 dan 2: tipe data

In [4]:
# CONTOH 3.1 — salin dulu, jangan sentuh yang mentah
d = mentah.copy()

jejak = []
def catat(langkah, terdampak, alasan):
    """Setiap perubahan dicatat SAAT dikerjakan, bukan belakangan."""
    jejak.append({'langkah': langkah, 'baris_terdampak': int(terdampak), 'alasan': alasan})
    print(f'  {langkah}: {int(terdampak)} baris')

print('Siap. Data mentah tetap utuh di variabel `mentah`.')

Siap. Data mentah tetap utuh di variabel `mentah`.


In [5]:
# CONTOH 3.2 — uang: dari teks ke angka
berformat = d['total_bayar'].astype(str).str.startswith('Rp')

d['total_bayar'] = pd.to_numeric(
    d['total_bayar'].astype(str)
      .str.replace('Rp', '', regex=False)
      .str.replace('.', '', regex=False)      # titik di sini pemisah ribuan, bukan desimal
      .str.strip(),
    errors='coerce')

catat('Konversi total_bayar ke angka', berformat.sum(),
      'Buang "Rp" dan pemisah ribuan agar kolom dapat dihitung')
print('Gagal dikonversi:', int(d['total_bayar'].isna().sum()))

  Konversi total_bayar ke angka: 250 baris
Gagal dikonversi: 0


> **Perhatikan `errors='coerce'`.** Nilai yang tidak bisa dikonversi menjadi `NaN` alih-alih menghentikan
> program. Itu sengaja: kita ingin tahu berapa banyak yang gagal, bukan berhenti di nilai pertama yang aneh.
> Selalu cetak jumlah kegagalannya. Kalau angkanya besar, berarti asumsi format Anda salah.

In [6]:
# CONTOH 3.3 — tanggal: tangani tiap format secara terpisah
t = d['tanggal_transaksi'].astype(str)
iso = t.str.match(r'^\d{4}-')

baru = pd.Series(pd.NaT, index=t.index, dtype='datetime64[ns]')
baru[iso]  = pd.to_datetime(t[iso],  format='%Y-%m-%d %H:%M:%S', errors='coerce')
baru[~iso] = pd.to_datetime(t[~iso], format='%d/%m/%Y %H:%M',    errors='coerce')
d['tanggal_transaksi'] = baru

catat('Konversi tanggal per format', len(d),
      'format="mixed" dengan dayfirst merusak 483 baris berformat ISO tanpa memunculkan galat')

print('Gagal   :', int(d['tanggal_transaksi'].isna().sum()))
print('Rentang :', d['tanggal_transaksi'].min(), 'sampai', d['tanggal_transaksi'].max())
print('Bulan   :', sorted(d['tanggal_transaksi'].dt.month.dropna().unique()))

  Konversi tanggal per format: 1543 baris
Gagal   : 0
Rentang : 2025-08-01 07:09:00 sampai 2025-11-28 16:25:00
Bulan   : [np.int32(8), np.int32(9), np.int32(10), np.int32(11)]


> Uji akal sehat dari pekan lalu dijalankan lagi di sini: empat bulan, Agustus sampai November. Benar.

---
## Bagian 4 — Langkah 2 sampai 4: teks, placeholder, kategori

In [7]:
# CONTOH 4.1 — buang spasi pada SELURUH kolom teks sekaligus
kolom_teks = [c for c in d.columns if d[c].dtype == object or str(d[c].dtype) == 'str']

n_spasi = 0
for kol in kolom_teks:
    s = d[kol].astype('string')
    n_spasi += int((s != s.str.strip()).sum())
    d[kol] = s.str.strip()

catat('Buang spasi awal dan akhir kolom teks', n_spasi,
      'Spasi tersembunyi memecah groupby dan melambungkan nunique')

  Buang spasi awal dan akhir kolom teks: 160 baris


In [8]:
# CONTOH 4.2 — placeholder menjadi kosong
CURIGA = ['n/a', 'na', '-', '', 'null', 'none', 'tidak diketahui', 'unknown', '?', '#n/a']

n_ph = 0
for kol in kolom_teks:
    s = d[kol].astype('string')
    m = s.str.lower().isin(CURIGA)
    n_ph += int(m.sum())
    d.loc[m, kol] = pd.NA
catat('Placeholder teks menjadi kosong', n_ph, 'N/A, -, null, dan teks kosong bukan nilai sungguhan')

n999 = int((d['rating_layanan'] == -999).sum())
d.loc[d['rating_layanan'] == -999, 'rating_layanan'] = np.nan
catat('Placeholder -999 pada rating menjadi kosong', n999,
      'Angka -999 merusak rata-rata tanpa pernah terdeteksi isna()')

  Placeholder teks menjadi kosong: 19 baris
  Placeholder -999 pada rating menjadi kosong: 49 baris


In [9]:
# CONTOH 4.3 — penyeragaman kategori, dengan peta keputusan yang eksplisit
# Bagian yang bisa dikerjakan mesin:
d['kategori'] = d['kategori'].astype('string').str.strip().str.title()

# Bagian yang HARUS diputuskan manusia, ditulis sebagai peta agar bisa ditinjau orang lain:
PETA_KATEGORI = {
    'Atk'        : 'ATK',          # penyeragaman penulisan
    'Alat Tulis' : 'ATK',          # sinonim; dipastikan ke pengurus koperasi
    'Minumam'    : 'Minuman',      # salah ketik yang jelas
}
sebelum = d['kategori'].nunique()
d['kategori'] = d['kategori'].replace(PETA_KATEGORI)

catat('Seragamkan kategori', sebelum - d['kategori'].nunique(),
      'Huruf dan spasi oleh kode; sinonim dan salah ketik diputuskan manual lalu dicatat di peta')
print(sorted(d['kategori'].dropna().unique()))

  Seragamkan kategori: 2 baris
['ATK', 'Jasa', 'Makanan', 'Minuman', 'Pulsa']


> **Mengapa peta ditulis sebagai dictionary, bukan rangkaian `replace` berantai?** Karena peta bisa dibaca
> orang lain, bisa ditinjau, bisa ditambah, dan ikut masuk ke laporan sebagai lampiran. Rangkaian `replace`
> yang tersebar di beberapa sel tidak bisa.
>
> Catatan pada `'Alat Tulis' : 'ATK'` bukan hiasan. Itu bukti bahwa Anda **bertanya**, bukan menebak.

In [10]:
# CONTOH 4.4 — kolom teks lain
d['metode_bayar'] = d['metode_bayar'].astype('string').str.strip().str.title().replace({'Qris': 'QRIS'})
d['kasir']        = d['kasir'].astype('string').str.strip().str.title()

print('metode_bayar:', sorted(d['metode_bayar'].dropna().unique()))
print('kasir       :', sorted(d['kasir'].dropna().unique()))

metode_bayar: ['QRIS', 'Saldo Koperasi', 'Transfer', 'Tunai']
kasir       : ['Bagus', 'Dimas', 'Rina', 'Sari']


### TUGAS 1 — hasil penyeragaman

In [14]:
# TUGAS 1
p5_kat      = d['kategori'].nunique()
p5_metode   = d['metode_bayar'].dropna().nunique()
p5_anggota  = d['id_anggota'].astype(str).str.strip().nunique()

cek('P5_kat', p5_kat)
cek('P5_metode', p5_metode)
cek('P5_anggota', p5_anggota)

BENAR   P5_kat = 5
BENAR   P5_metode = 4
BENAR   P5_anggota = 259


---
## Bagian 5 — Langkah 5: duplikat

In [15]:
# CONTOH 5.1 — duplikat persis
n_persis = int(d.duplicated().sum())
d = d.drop_duplicates()
catat('Hapus duplikat persis', n_persis, 'Seluruh kolom identik; satu baris disimpan')

  Hapus duplikat persis: 39 baris


> Jumlahnya **lebih besar** daripada yang Anda temukan pekan lalu. Sebabnya: setelah teks diseragamkan,
> baris yang tadinya berbeda hanya pada kapitalisasi kini menjadi benar-benar identik.
>
> Ini contoh mengapa urutan penting. Membersihkan teks lebih dulu membuat duplikat nyaris berubah
> menjadi duplikat persis, sehingga bisa ditangani otomatis.

In [16]:
# CONTOH 5.2 — duplikat yang tersisa pada kunci utama
n_id = int(d['id_transaksi'].duplicated(keep=False).sum())
print('Baris dengan id_transaksi berulang:', n_id)

# Aturan penyatuan: simpan baris yang catatan_kasir-nya terisi.
d = (d.sort_values('catatan_kasir', na_position='last')
       .drop_duplicates(subset=['id_transaksi'], keep='first'))
catat('Satukan duplikat menurut id_transaksi', n_id,
      'Satu transaksi hanya boleh satu baris; dipilih baris yang catatannya terisi')

Baris dengan id_transaksi berulang: 15
  Satukan duplikat menurut id_transaksi: 15 baris


In [17]:
# CONTOH 5.3 — kunci utama yang kosong
n_kosong = int(d['id_transaksi'].isna().sum())
d = d[d['id_transaksi'].notna()]
catat('Buang baris tanpa id_transaksi', n_kosong,
      'Kunci utama kosong tidak dapat ditelusuri kembali ke sumber')

  Buang baris tanpa id_transaksi: 1 baris


---
## Bagian 6 — Langkah 6: nilai mustahil

Pohon keputusan yang dipakai di mata kuliah ini:

```
Nilai ini mustahil.
│
├─ Tahu nilai yang benar?            -> perbaiki, lalu catat sumber kebenarannya
├─ Tahu pasti ini kesalahan input?   -> jadikan kosong
├─ Mungkin sah tetapi belum jelas?   -> biarkan, tambahkan kolom penanda
└─ Barisnya tidak bisa dipakai       -> buang, lalu catat berapa banyak
```

Yang **tidak** ada di pohon ini: menebak. Kalau Anda tidak tahu, jangan mengarang nilai.

In [18]:
# CONTOH 6.1 — jumlah nol atau negatif: dijadikan kosong, bukan dibuang
n_j = int((d['jumlah'] <= 0).sum())
d.loc[d['jumlah'] <= 0, 'jumlah'] = np.nan
catat('jumlah <= 0 menjadi kosong', n_j,
      'Belum jelas retur atau salah input; barisnya masih berguna untuk kolom lain')

  jumlah <= 0 menjadi kosong: 29 baris


In [19]:
# CONTOH 6.2 — diskon di luar rentang
buruk = (d['diskon_persen'] < 0) | (d['diskon_persen'] > 100)
n_dk = int(buruk.sum())
d.loc[buruk, 'diskon_persen'] = np.nan
catat('Diskon di luar 0-100 menjadi kosong', n_dk, 'Nilai mustahil; menunggu konfirmasi pengurus')

  Diskon di luar 0-100 menjadi kosong: 9 baris


In [20]:
# CONTOH 6.3 — pelanggaran aturan: DITANDAI, bukan ditimpa
d['total_seharusnya']   = (d['jumlah'] * d['harga_satuan'] * (1 - d['diskon_persen'] / 100)).round()
d['selisih_total']      = d['total_bayar'] - d['total_seharusnya']
d['total_mencurigakan'] = d['selisih_total'].abs() > 1

catat('Tandai total yang tidak cocok', d['total_mencurigakan'].sum(),
      'Ditandai, tidak ditimpa, karena belum diketahui kolom mana yang keliru')

d.loc[d['total_mencurigakan'], ['jumlah','harga_satuan','diskon_persen','total_bayar','total_seharusnya']].head(5)

  Tandai total yang tidak cocok: 50 baris


,jumlah,harga_satuan,diskon_persen,total_bayar,total_seharusnya
1381,4.0,5000,0.0,200000,20000.0
1401,3.0,6000,5.0,8550,17100.0
148,7.0,5000,0.0,350000,35000.0
897,1.0,35000,0.0,350000,35000.0
1483,2.0,6000,0.0,24000,12000.0


> **Mengapa ditandai, bukan diperbaiki?** Karena kita tidak tahu kolom mana yang salah. Bisa `total_bayar`
> yang keliru, bisa `jumlah`, bisa `harga_satuan`. Menimpa `total_bayar` dengan hasil hitungan sama saja
> dengan memutuskan bahwa tiga kolom lain pasti benar — tanpa dasar apa pun.
>
> Kolom penanda memindahkan keputusan ke tahap analisis, di mana dampaknya bisa diuji:
> jalankan analisis dengan dan tanpa baris bertanda, lalu lihat apakah kesimpulannya berubah.

---
## Bagian 7 — Langkah 7: nilai kosong, dan mengapa tidak ada pilihan yang netral

Empat strategi yang lazim dipakai:

| Strategi | Caranya | Cocok untuk |
|---|---|---|
| Buang barisnya | `dropna()` | MCAR, dan hanya bila yang hilang sedikit |
| Isi nilai tengah | `fillna(mean)` atau `fillna(median)` | MCAR, kolom yang bukan fokus analisis |
| Isi per kelompok | `groupby().transform()` | MAR, bila kelompoknya memang penjelasnya |
| Tandai sebagai kategori | `fillna('Tidak tercatat')` | Kolom kategorikal; kekosongan itu sendiri bermakna |

Pekan lalu kita menduga `saldo_simpanan` bersifat **MNAR** — nilai besar lebih sering disembunyikan.
Sekarang kita uji dugaan itu.

### Keistimewaan data latihan

Dataset ini sintetis, dan itu memberi kita satu kemampuan yang **tidak pernah Anda miliki pada data nyata**:
kita bisa melihat nilai yang sebenarnya disembunyikan, lalu mengukur seberapa jauh setiap strategi meleset.

Berkas `P05_nilai_sebenarnya.csv` berisi nilai asli sebelum dikosongkan. Pada proyek Anda berkas semacam ini
tidak akan pernah ada — karena itu jalankan eksperimen ini sekarang, selagi bisa.

In [21]:
# CONTOH 7.1 — eksperimen imputasi
if os.path.exists('P05_nilai_sebenarnya.csv'):
    truth = pd.read_csv('P05_nilai_sebenarnya.csv')
    asli = truth['_saldo_asli']
    obs  = truth['saldo_simpanan']

    hasil = pd.DataFrame({
        'KEBENARAN (semua anggota)' : asli.describe()[['mean','50%','std','max']],
        'Hanya yang terlihat'       : obs.dropna().describe()[['mean','50%','std','max']],
        'Diisi dengan mean'         : obs.fillna(obs.mean()).describe()[['mean','50%','std','max']],
        'Diisi dengan median'       : obs.fillna(obs.median()).describe()[['mean','50%','std','max']],
    }).T.round(0)
    hasil.columns = ['mean', 'median', 'std', 'max']
    display(hasil.map(lambda v: f'{v:,.0f}'))
else:
    print('Berkas kebenaran tidak tersedia; lewati sel ini.')

Berkas kebenaran tidak tersedia; lewati sel ini.


In [22]:
# CONTOH 7.2 — nilai yang disembunyikan itu seperti apa?
if os.path.exists('P05_nilai_sebenarnya.csv'):
    m = obs.isna()
    print('Saldo yang TERLIHAT     — rata-rata:', f'{obs.dropna().mean():>10,.0f}')
    print('Saldo yang DISEMBUNYIKAN— rata-rata:', f'{asli[m].mean():>10,.0f}')
    print()
    print('Rasio:', round(asli[m].mean() / obs.dropna().mean(), 2), 'kali lipat')
    print()
    for nama, seri in [('Buang barisnya', obs.dropna()),
                       ('Isi dengan mean', obs.fillna(obs.mean())),
                       ('Isi dengan median', obs.fillna(obs.median()))]:
        salah = abs(seri.mean() - asli.mean()) / asli.mean() * 100
        print(f'{nama:20s} meleset {salah:5.1f}% dari rata-rata sebenarnya')

> **Baca angka-angka itu pelan-pelan.**
>
> Saldo yang disembunyikan rata-ratanya sekitar **2,8 kali** saldo yang terlihat. Saldo tertinggi yang
> terlihat sekitar 3,5 juta, padahal yang sebenarnya ada mencapai 7,2 juta.
>
> Dan yang paling penting: **ketiga strategi sama-sama meleset 15 sampai 18%.** Membuang baris
> tidak lebih aman daripada mengisi. Keduanya sama-sama menghapus bagian atas sebaran.
>
> Pada data MNAR, tidak ada teknik statistik yang bisa menyelamatkan Anda. Yang bisa dilakukan hanya dua:
> **kembali ke sumber data untuk melengkapi**, atau **melaporkan keterbatasannya secara terbuka** —
> misalnya menulis "rata-rata saldo anggota minimal Rp336.000; angka sebenarnya lebih tinggi karena
> anggota bersaldo besar tidak menampilkan saldonya".
>
> Kalimat seperti itu, di laporan Anda, lebih bernilai daripada angka mana pun.

In [23]:
# CONTOH 7.3 — keputusan kita untuk dataset ini
# saldo_simpanan: TIDAK diimputasi. Dibiarkan kosong dan keterbatasannya dilaporkan.
catat('saldo_simpanan dibiarkan kosong', d['saldo_simpanan'].isna().sum(),
      'Diduga MNAR; imputasi apa pun akan membuat rata-rata saldo tampak lebih kecil daripada kenyataan')

# berat_kg: MAR, dan kekosongannya BERMAKNA (barang non-fisik memang tak punya berat).
d['berlaku_berat'] = ~d['kategori'].isin(['Pulsa', 'Jasa'])
catat('Tandai keberlakuan berat_kg', (~d['berlaku_berat']).sum(),
      'Kosong karena tidak berlaku, bukan karena hilang; diberi kolom penanda, tidak diisi')

# catatan_kasir: kekosongan itu sendiri sebuah kategori.
d['ada_catatan'] = d['catatan_kasir'].notna()
catat('Tandai keberadaan catatan kasir', (~d['ada_catatan']).sum(),
      'Kosong berarti kasir tidak menulis apa pun; dijadikan penanda biner')

  saldo_simpanan dibiarkan kosong: 156 baris
  Tandai keberlakuan berat_kg: 290 baris
  Tandai keberadaan catatan kasir: 1004 baris


### TUGAS 2 — hasil pembersihan

In [24]:
# TUGAS 2
d = d.reset_index(drop=True)

p5_baris_bersih  = len(d)
p5_dibuang       = len(mentah) - len(d)
p5_curiga        = int(d['total_mencurigakan'].sum())
p5_rating_kosong = int(d['rating_layanan'].isna().sum())

cek('P5_baris_bersih', p5_baris_bersih)
cek('P5_dibuang', p5_dibuang)
cek('P5_curiga', p5_curiga)
cek('P5_rating_kosong', p5_rating_kosong)

BENAR   P5_baris_bersih = 1493
BENAR   P5_dibuang = 50
BENAR   P5_curiga = 50
BENAR   P5_rating_kosong = 46


---
## Bagian 8 — Kolom turunan dan encoding

In [25]:
# CONTOH 8.1 — kolom turunan
d['bulan']        = d['tanggal_transaksi'].dt.to_period('M').astype(str)
d['hari']         = d['tanggal_transaksi'].dt.day_name()
d['jam']          = d['tanggal_transaksi'].dt.hour
d['nilai_bersih'] = d['total_bayar']

d[['tanggal_transaksi', 'bulan', 'hari', 'jam']].head(3)

,tanggal_transaksi,bulan,hari,jam
0,2025-08-27 09:43:00,2025-08,Wednesday,9
1,2025-10-09 16:57:00,2025-10,Thursday,16
2,2025-08-27 09:20:00,2025-08,Wednesday,9


In [26]:
# CONTOH 8.2 — binning: angka menjadi kelompok
d['kelas_nilai'] = pd.cut(d['total_bayar'],
                          bins=[0, 10_000, 30_000, 75_000, np.inf],
                          labels=['Sangat kecil', 'Kecil', 'Sedang', 'Besar'])
print(d['kelas_nilai'].value_counts().sort_index())

kelas_nilai
Sangat kecil    299
Kecil           620
Sedang          511
Besar            63
Name: count, dtype: int64


> **Batas bin adalah keputusan, bukan perhitungan.** Angka 10.000, 30.000, dan 75.000 di atas berasal dari
> pengetahuan tentang harga barang koperasi, bukan dari rumus. Tuliskan alasannya di *cleaning log*, karena
> mengubah batas bin akan mengubah kesimpulan analisis Anda.

In [27]:
# CONTOH 8.3 — encoding, dan kapan masing-masing dipakai
# Ordinal: ada urutan, jadi pakai kategori berurutan
urut = ['Sangat kecil', 'Kecil', 'Sedang', 'Besar']
d['kelas_nilai'] = pd.Categorical(d['kelas_nilai'], categories=urut, ordered=True)

# Nominal: tidak ada urutan, jadi one-hot bila nanti dibutuhkan
contoh_onehot = pd.get_dummies(d['metode_bayar'], prefix='bayar', dummy_na=True)
print(contoh_onehot.columns.tolist())
print()
print('Untuk EDA, one-hot jarang diperlukan. Simpan untuk tahap pemodelan.')

['bayar_QRIS', 'bayar_Saldo Koperasi', 'bayar_Transfer', 'bayar_Tunai', 'bayar_<NA>']

Untuk EDA, one-hot jarang diperlukan. Simpan untuk tahap pemodelan.


> **Jangan melakukan encoding hanya karena bisa.** Pada EDA, kategori lebih mudah dibaca daripada kolom
> nol-satu. One-hot baru berguna saat masuk pemodelan, yang bukan wilayah mata kuliah ini.

---
## Bagian 9 — Menyusun ulang bentuk tabel

**Tidy data** berarti: satu baris satu observasi, satu kolom satu variabel, satu tabel satu satuan amatan.

In [28]:
# CONTOH 9.1 — pivot: dari panjang ke lebar (untuk dibaca manusia)
lebar = (d.pivot_table(index='bulan', columns='kategori', values='total_bayar',
                       aggfunc='sum', observed=True)
          .fillna(0).round(0).astype(int))
lebar

kategori,ATK,Jasa,Makanan,Minuman,Pulsa
bulan,,,,,
2025-08,1498800,901450,5904300,3410675,1828700
2025-09,1251250,824200,5461800,3289175,1417500
2025-10,1337150,1299400,5087800,2817275,1583250
2025-11,1005100,753950,5648750,2571100,1253200


In [29]:
# CONTOH 9.2 — melt: kembali ke panjang (untuk dianalisis mesin)
panjang = lebar.reset_index().melt(id_vars='bulan', var_name='kategori', value_name='total')
print(panjang.shape)
panjang.head(6)

(20, 3)


,bulan,kategori,total
0,2025-08,ATK,1498800
1,2025-09,ATK,1251250
2,2025-10,ATK,1337150
3,2025-11,ATK,1005100
4,2025-08,Jasa,901450
5,2025-09,Jasa,824200


> **Kapan memakai yang mana.** Bentuk **lebar** enak dibaca manusia dan cocok untuk tabel di laporan.
> Bentuk **panjang** adalah yang dibutuhkan hampir semua fungsi grafik dan agregasi. Simpan data Anda
> dalam bentuk panjang, lalu ubah ke lebar hanya pada saat menampilkan.

In [30]:
# CONTOH 9.3 — merge: menggabungkan tabel lain
# Tabel ringkasan per anggota, lalu digabungkan kembali ke data transaksi.
profil_anggota = (d.groupby('id_anggota')
                    .agg(jumlah_transaksi=('id_transaksi', 'count'),
                         total_belanja=('total_bayar', 'sum'))
                    .reset_index())

gabung = d.merge(profil_anggota, on='id_anggota', how='left', validate='many_to_one')
print('Sebelum:', d.shape, '-> sesudah:', gabung.shape)
profil_anggota.head(3)

Sebelum: (1493, 26) -> sesudah: (1493, 28)


,id_anggota,jumlah_transaksi,total_belanja
0,AGT1001,6,190000
1,AGT1002,8,258600
2,AGT1003,5,108000


> **`validate='many_to_one'` itu rem darurat.** Tanpa itu, kunci yang ternyata tidak unik akan membuat
> baris berlipat ganda tanpa peringatan, dan total belanja Anda membengkak. Selalu nyatakan hubungan yang
> Anda harapkan; biarkan pandas yang menolak bila ternyata keliru.
>
> Periksa juga jumlah baris sebelum dan sesudah `merge`. Kalau bertambah, ada yang salah.

---
## Bagian 10 — Pipeline yang dapat dijalankan ulang

Sel-sel yang berserakan tidak bisa dijalankan ulang dengan hasil yang sama. Satukan menjadi satu fungsi.

In [38]:
# CONTOH 10.1 — kerangka pipeline lengkap
def bersihkan(df_mentah):
    """Terima data mentah, kembalikan data bersih beserta jejak keputusannya."""
    x = df_mentah.copy()
    jejak_log = []

    def c(langkah, n, alasan):
        jejak_log.append({'langkah': langkah, 'baris_terdampak': int(n), 'alasan': alasan})

    # 1. Tipe Data: total_bayar & tanggal_transaksi
    berformat = x['total_bayar'].astype(str).str.startswith('Rp')
    x['total_bayar'] = pd.to_numeric(
        x['total_bayar'].astype(str)
          .str.replace('Rp', '', regex=False)
          .str.replace('.', '', regex=False)
          .str.strip(),
        errors='coerce')
    c('Konversi total_bayar ke angka', berformat.sum(), 'Buang "Rp" dan pemisah ribuan agar kolom dapat dihitung')

    t = x['tanggal_transaksi'].astype(str)
    iso = t.str.match(r'^\d{4}-')
    baru = pd.Series(pd.NaT, index=t.index, dtype='datetime64[ns]')
    baru[iso]  = pd.to_datetime(t[iso],  format='%Y-%m-%d %H:%M:%S', errors='coerce')
    baru[~iso] = pd.to_datetime(t[~iso], format='%d/%m/%Y %H:%M',    errors='coerce')
    x['tanggal_transaksi'] = baru
    c('Konversi tanggal per format', len(x), 'format="mixed" merusak data ISO; ditangani terpisah')

    # 2. Spasi pada kolom teks
    kolom_teks = [col for col in x.columns if x[col].dtype == object or str(x[col].dtype) == 'str']
    n_spasi = 0
    for kol in kolom_teks:
        s = x[kol].astype('string')
        n_spasi += int((s != s.str.strip()).sum())
        x[kol] = s.str.strip()
    c('Buang spasi awal dan akhir kolom teks', n_spasi, 'Spasi tersembunyi memecah groupby dan nunique')

    # 3. Placeholder menjadi NaN
    CURIGA = ['n/a', 'na', '-', '', 'null', 'none', 'tidak diketahui', 'unknown', '?', '#n/a']
    n_ph = 0
    for kol in kolom_teks:
        s = x[kol].astype('string')
        m = s.str.lower().isin(CURIGA)
        n_ph += int(m.sum())
        x.loc[m, kol] = pd.NA
    c('Placeholder teks menjadi kosong', n_ph, 'N/A, -, null, dan teks kosong bukan nilai sungguhan')

    n999 = int((x['rating_layanan'] == -999).sum())
    x.loc[x['rating_layanan'] == -999, 'rating_layanan'] = np.nan
    c('Placeholder -999 pada rating menjadi kosong', n999, 'Angka -999 merusak rata-rata tanpa terdeteksi isna()')

    # 4. Kategori & Teks
    x['kategori'] = x['kategori'].astype('string').str.strip().str.title()
    PETA_KATEGORI = {'Atk': 'ATK', 'Alat Tulis': 'ATK', 'Minumam': 'Minuman'}
    sebelum_kat = x['kategori'].nunique()
    x['kategori'] = x['kategori'].replace(PETA_KATEGORI)
    c('Seragamkan kategori', sebelum_kat - x['kategori'].nunique(), 'Huruf dan spasi diseragamkan; sinonim disatukan')

    x['metode_bayar'] = x['metode_bayar'].astype('string').str.strip().str.title().replace({'Qris': 'QRIS'})
    x['kasir']        = x['kasir'].astype('string').str.strip().str.title()

    # 5. Duplikat
    n_persis = int(x.duplicated().sum())
    x = x.drop_duplicates()
    c('Hapus duplikat persis', n_persis, 'Seluruh kolom identik; satu baris disimpan')

    n_id = int(x['id_transaksi'].duplicated(keep=False).sum())
    x = (x.sort_values('catatan_kasir', na_position='last')
           .drop_duplicates(subset=['id_transaksi'], keep='first'))
    c('Satukan duplikat menurut id_transaksi', n_id, 'Pilih baris yang catatan_kasir-nya terisi')

    n_kosong = int(x['id_transaksi'].isna().sum())
    x = x[x['id_transaksi'].notna()]
    c('Buang baris tanpa id_transaksi', n_kosong, 'Kunci utama kosong tidak dapat ditelusuri kembali')

    # 6. Nilai Mustahil
    n_j = int((x['jumlah'] <= 0).sum())
    x.loc[x['jumlah'] <= 0, 'jumlah'] = np.nan
    c('jumlah <= 0 menjadi kosong', n_j, 'Belum jelas retur atau salah input; baris disimpan')

    buruk = (x['diskon_persen'] < 0) | (x['diskon_persen'] > 100)
    n_dk = int(buruk.sum())
    x.loc[buruk, 'diskon_persen'] = np.nan
    c('Diskon di luar 0-100 menjadi kosong', n_dk, 'Nilai mustahil; diubah ke NaN')

    x['total_seharusnya']   = (x['jumlah'] * x['harga_satuan'] * (1 - x['diskon_persen'] / 100)).round()
    x['selisih_total']      = x['total_bayar'] - x['total_seharusnya']
    x['total_mencurigakan'] = x['selisih_total'].abs() > 1
    c('Tandai total yang tidak cocok', int(x['total_mencurigakan'].sum()), 'Ditandai, tidak ditimpa')

    # 7. Nilai Kosong
    c('saldo_simpanan dibiarkan kosong', int(x['saldo_simpanan'].isna().sum()), 'Diduga MNAR; tidak diimputasi')
    x['berlaku_berat'] = ~x['kategori'].isin(['Pulsa', 'Jasa'])
    c('Tandai keberlakuan berat_kg', int((~x['berlaku_berat']).sum()), 'Kosong karena barang non-fisik')
    x['ada_catatan'] = x['catatan_kasir'].notna()
    c('Tandai keberadaan catatan kasir', int((~x['ada_catatan']).sum()), 'Kosong diubah jadi penanda biner')

    # 8. Kolom Turunan
    x['bulan']        = x['tanggal_transaksi'].dt.to_period('M').astype(str)
    x['hari']         = x['tanggal_transaksi'].dt.day_name()
    x['jam']          = x['tanggal_transaksi'].dt.hour
    x['nilai_bersih'] = x['total_bayar']
    x['kelas_nilai']  = pd.cut(x['total_bayar'],
                              bins=[0, 10_000, 30_000, 75_000, np.inf],
                              labels=['Sangat kecil', 'Kecil', 'Sedang', 'Besar'])

    return x.reset_index(drop=True), pd.DataFrame(jejak_log)

print('Fungsi pipeline bersihkan() siap digunakan.')

Fungsi pipeline bersihkan() siap digunakan.


> **Uji reproduktibilitas yang murah:** jalankan `bersihkan(mentah)` dua kali, lalu bandingkan hasilnya.
> Kalau berbeda, ada yang bergantung pada urutan acak atau pada keadaan sel sebelumnya. Pada pertemuan 15
> tim lain akan menjalankan ulang notebook Anda, jadi masalah ini lebih murah diselesaikan sekarang.

In [32]:
# CONTOH 10.2 — simpan hasil
os.makedirs('data/processed', exist_ok=True)
os.makedirs('reports', exist_ok=True)

d.to_csv('data/processed/transaksi_bersih.csv', index=False)
print('Tersimpan:', d.shape)
print()
print('Data mentah tidak disentuh:', mentah.shape)

Tersimpan: (1493, 26)

Data mentah tidak disentuh: (1543, 16)


---
## Bagian 11 — Cleaning log

Dokumen ini yang membedakan pembersihan yang bisa dipertanggungjawabkan dari pembersihan yang asal jalan.
Setiap baris menjawab tiga hal: **apa yang diubah, berapa banyak, dan mengapa**.

In [33]:
# CONTOH 11.1 — cetak log
log = pd.DataFrame(jejak)
log.index = range(1, len(log) + 1)
log

,langkah,baris_terdampak,alasan
1,Konversi total_bayar ke angka,250,"Buang ""Rp"" dan pemisah ribuan agar kolom dapat..."
2,Konversi tanggal per format,1543,"format=""mixed"" dengan dayfirst merusak 483 bar..."
3,Buang spasi awal dan akhir kolom teks,160,Spasi tersembunyi memecah groupby dan melambun...
4,Placeholder teks menjadi kosong,19,"N/A, -, null, dan teks kosong bukan nilai sung..."
5,Placeholder -999 pada rating menjadi kosong,49,Angka -999 merusak rata-rata tanpa pernah terd...
6,Seragamkan kategori,2,Huruf dan spasi oleh kode; sinonim dan salah k...
7,Hapus duplikat persis,39,Seluruh kolom identik; satu baris disimpan
8,Satukan duplikat menurut id_transaksi,15,Satu transaksi hanya boleh satu baris; dipilih...
9,Buang baris tanpa id_transaksi,1,Kunci utama kosong tidak dapat ditelusuri kemb...
10,jumlah <= 0 menjadi kosong,29,Belum jelas retur atau salah input; barisnya m...


In [34]:
# CONTOH 11.2 — simpan sebagai Markdown untuk repositori
baris = ['# Cleaning Log', '',
         'Dataset: transaksi_koperasi_2025.csv  ',
         f'Baris mentah: {len(mentah)} -> baris bersih: {len(d)}  ',
         '', '| # | Langkah | Baris terdampak | Alasan |', '|---|---|---|---|']
for i, r in log.iterrows():
    baris.append(f"| {i} | {r['langkah']} | {r['baris_terdampak']} | {r['alasan']} |")

with open('cleaning_log.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(baris))

print('\n'.join(baris[:10]))

# Cleaning Log

Dataset: transaksi_koperasi_2025.csv  
Baris mentah: 1543 -> baris bersih: 1493  

| # | Langkah | Baris terdampak | Alasan |
|---|---|---|---|
| 1 | Konversi total_bayar ke angka | 250 | Buang "Rp" dan pemisah ribuan agar kolom dapat dihitung |
| 2 | Konversi tanggal per format | 1543 | format="mixed" dengan dayfirst merusak 483 baris berformat ISO tanpa memunculkan galat |
| 3 | Buang spasi awal dan akhir kolom teks | 160 | Spasi tersembunyi memecah groupby dan melambungkan nunique |


### TUGAS 3 — kelengkapan log

In [35]:
# TUGAS 3
p5_langkah   = len(jejak)
p5_total_num = int(d['total_bayar'].notna().sum())
p5_bulan     = d['tanggal_transaksi'].dt.to_period('M').nunique()

cek('P5_langkah', p5_langkah)
cek('P5_total_num', p5_total_num)
cek('P5_bulan', p5_bulan)

BENAR   P5_langkah = 15
BENAR   P5_total_num = 1493
BENAR   P5_bulan = 4


> Kalau jumlah langkah Anda berbeda dari kunci, itu **tidak otomatis salah** — tim boleh memecah atau
> menggabungkan langkah. Yang penting setiap perubahan ada catatannya. Angka pada kunci mengikuti
> pembagian langkah di workbook ini.

---
## Bagian 12 — Verifikasi: jalankan ulang diagnosis pertemuan 4

Pembersihan belum selesai sampai Anda membuktikan bahwa masalahnya benar-benar hilang —
dan bahwa tidak ada masalah baru yang Anda ciptakan sendiri.

In [36]:
# CONTOH 12.1 — diagnosis ulang
def diagnosa_singkat(data, nama):
    print(f'=== {nama} ===')
    print('  Baris                     :', len(data))
    print('  Duplikat persis           :', int(data.duplicated().sum()))
    print('  Kategori unik             :', data['kategori'].astype(str).nunique())
    print('  id_anggota unik           :', data['id_anggota'].astype(str).nunique())
    print('  total_bayar bertipe angka :', pd.api.types.is_numeric_dtype(data['total_bayar']))
    print('  rating bernilai -999      :', int((data['rating_layanan'] == -999).sum()))
    print('  jumlah <= 0               :', int((data['jumlah'] <= 0).sum()))
    print()

diagnosa_singkat(mentah, 'SEBELUM')
diagnosa_singkat(d, 'SESUDAH')

=== SEBELUM ===
  Baris                     : 1543
  Duplikat persis           : 24
  Kategori unik             : 18
  id_anggota unik           : 318
  total_bayar bertipe angka : False
  rating bernilai -999      : 49
  jumlah <= 0               : 29

=== SESUDAH ===
  Baris                     : 1493
  Duplikat persis           : 0
  Kategori unik             : 5
  id_anggota unik           : 259
  total_bayar bertipe angka : True
  rating bernilai -999      : 0
  jumlah <= 0               : 0



In [37]:
# CONTOH 12.2 — apa yang HILANG dan apa yang BERTAMBAH
banding = pd.DataFrame({
    'kosong_sebelum': mentah.isna().sum(),
    'kosong_sesudah': d.reindex(columns=mentah.columns).isna().sum(),
})
banding['selisih'] = banding['kosong_sesudah'] - banding['kosong_sebelum']
banding[banding['selisih'] != 0].sort_values('selisih', ascending=False)

,kosong_sebelum,kosong_sesudah,selisih
rating_layanan,0,46,46
jumlah,0,29,29
metode_bayar,9,28,19
diskon_persen,0,9,9
saldo_simpanan,159,156,-3
id_transaksi,7,0,-7
berat_kg,351,341,-10
catatan_kasir,1044,1004,-40


> **Nilai kosong Anda bertambah, dan itu wajar.** Placeholder yang tadinya menyamar sebagai angka kini
> jujur mengaku kosong. Data yang lebih jujur memang terlihat lebih buruk, dan itulah tandanya pembersihan
> Anda berhasil.
>
> Laporkan angka ini apa adanya di M2. Laporan yang mengaku "nilai kosong bertambah dari 1.570 menjadi
> sekian karena placeholder diungkap" jauh lebih meyakinkan daripada laporan yang semuanya nol.

---
## Bagian 13 — Kerjakan untuk proyek tim: M2

**M2 — Data Quality Report, bobot 10%.** Tiga berkas yang dikumpulkan:

| Berkas | Isi | Diperiksa oleh rubrik |
|---|---|---|
| `reports/data_quality_report.docx` | Temuan, keputusan, keterbatasan | Kelengkapan diagnosis (25), Justifikasi (30) |
| `notebooks/02_cleaning.ipynb` | Pipeline yang dapat dijalankan ulang | Ketepatan implementasi (20) |
| `cleaning_log.md` | Setiap langkah, jumlah, dan alasan | Dokumentasi dan keterlacakan (15) |

Dimensi kelima, **Penanganan data pribadi (10)**, dinilai dari apakah data olahan Anda tetap aman —
yaitu lanjutan pekerjaan pertemuan 3.

Gunakan `TEMPLATE_Data_Quality_Report.docx` sebagai kerangka laporan.

In [39]:
# 13.1 — salin pola workbook ini ke dataset tim
# df_tim = pd.read_csv('data/raw/....csv')
# bersih_tim, log_tim = bersihkan(df_tim)
# bersih_tim.to_csv('data/processed/....csv', index=False)


In [40]:
# 13.2 — jalankan pemeriksa kelengkapan sebelum mengumpulkan
# Dari terminal, di dalam folder repositori tim:
#     python cek_m2.py
# Skrip akan memeriksa keberadaan berkas, struktur folder, dan isi minimal cleaning log.


### Daftar periksa M2

- [ ] Seluruh temuan pertemuan 4 muncul di laporan, termasuk yang Anda putuskan untuk dibiarkan
- [ ] Setiap keputusan punya alasan tertulis, bukan sekadar deskripsi tindakan
- [ ] Minimal satu keputusan menyebutkan **alternatif yang dipertimbangkan dan ditolak**
- [ ] Dampak terhadap sebaran data ditunjukkan untuk minimal satu kolom (pola Bagian 7)
- [ ] Keterbatasan dinyatakan terbuka, termasuk kolom yang tidak bisa diselamatkan
- [ ] `data/raw/` tidak berubah sama sekali dan tidak ikut diunggah
- [ ] Notebook berjalan dari awal sampai akhir tanpa galat setelah *Restart and run all*
- [ ] `cleaning_log.md` ada di akar repositori
- [ ] `python cek_m2.py` berjalan tanpa keluhan

> **Kesalahan yang paling mahal pada M2:** menulis apa yang dilakukan, bukan mengapa.
> "Kolom X diisi dengan median" adalah deskripsi. "Kolom X diisi dengan median karena kekosongannya
> tersebar merata di semua kelompok dan hanya 3%, sehingga pilihan ini tidak menggeser sebaran lebih dari
> 1%; alternatif membuang baris ditolak karena akan menghapus 48 transaksi yang kolom lainnya lengkap"
> adalah justifikasi. Rubrik memberi bobot 30 untuk yang kedua.

---
## Kalau Anda mentok

| Urutan | Yang dilakukan |
|---|---|
| 1 | Baca baris terakhir pesan galat |
| 2 | *Runtime → Restart and run all*. Banyak galat muncul karena sel dijalankan tidak berurutan. |
| 3 | Cetak `d.dtypes` dan `d.shape` sebelum serta sesudah langkah yang bermasalah |
| 4 | Tanyakan ke anggota tim |
| 5 | Tanyakan ke asisten saat sesi berlangsung |
| 6 | Tulis di forum tanya-jawab LMS; dosen menjawab setiap hari kerja |

**Galat yang paling sering muncul di pertemuan ini:**

| Pesan | Artinya | Perbaikannya |
|---|---|---|
| `can only use .str accessor with string values` | Kolomnya bukan teks | `.astype(str)` atau `.astype('string')` lebih dulu |
| `SettingWithCopyWarning` | Menulis ke salinan, bukan ke tabel aslinya | Pakai `.loc[baris, kolom] = nilai`, dan `.copy()` setelah menyaring |
| `could not convert string to float` | Masih ada teks di kolom angka | Pakai `errors='coerce'`, lalu hitung berapa yang gagal |
| `MergeError: Merge keys are not unique` | Kunci gabungan tidak unik | Itu justru gunanya `validate=`; perbaiki kuncinya, jangan hapus validasinya |

---
## Ringkasan pertemuan 5

1. **Pembersihan adalah rangkaian keputusan.** Yang dinilai adalah alasannya, bukan pilihannya.
2. **Urutan menentukan hasil.** Tipe data dulu, nilai kosong paling akhir.
3. **Menandai lebih aman daripada menebak.** Kolom penanda memindahkan keputusan ke tahap yang tepat.
4. **Pada data MNAR, semua strategi meleset** — termasuk membuang baris. Yang menolong hanya kembali ke
   sumber data, atau melaporkan keterbatasannya secara terbuka.
5. **Nilai kosong yang bertambah adalah tanda keberhasilan**, bukan kegagalan.
6. **Pembersihan belum selesai sampai diagnosisnya dijalankan ulang.**

### Setelah pertemuan 5

- Selesaikan dan kumpulkan **M2** sesuai tenggat di LMS.
- Kerjakan kuis formatif pertemuan 5.
- Pertemuan 6 kembali bersama dosen: analisis univariat, *outlier*, dan visualisasi yang jujur.